In [1]:
import pandas as pd
import numpy as np
from iteround import saferound

In [2]:
# load mazsim race scenario results
with pd.HDFStore('data/mazsim_results/results_2.h5') as store:
    h = store['/2050/households']
    p = store['/2050/persons']

mazs = h['maz_id']
p = p.drop(columns=['maz_id'])
p['maz_id'] = p['household_id'].map(mazs)

In [3]:
# Map MAZs to TAZs and add TAZ column to persons dataframe
blk = pd.read_csv('data/mazsim_results/blocks_2020_2026-10-07.csv')
tazs = blk.set_index('maz_id')['zone_id']
p['taz'] = p['maz_id'].map(tazs)

In [5]:
# Calculate the proportion of each race category within each TAZ

race_categories = {
    'white_hispanic': '(is_hispanic == 1) & (race == 1)',
    'non_white_hispanic': '(is_hispanic == 1) & (race != 1)',
    'white_non_hispanic': '(is_hispanic == 0) & (race == 1)',
    'black_non_hispanic': '(is_hispanic == 0) & (race == 2)',
    'other_non_hispanic': '(is_hispanic == 0) & (race.isin([3, 4, 5, 7, 8]))',
    'asian_non_hispanic': '(is_hispanic == 0) & (race == 6)',
    'two_or_more_races_non_hispanic': '(is_hispanic == 0) & (race == 9)'
}

race_cat_list = [category for category in race_categories.keys()]

for category, condition in race_categories.items():
    p.eval(f"{category} = {condition}", inplace=True)

p_sum = p.groupby('taz')[[category for category in race_categories.keys()]].sum()
taz_sum = p_sum.sum(axis=1)
p_prop = p_sum.div(taz_sum, axis=0)

In [9]:
# load in taz popsim controls
controls = pd.read_csv('output/data/future_controls.csv').dropna(subset=['taz_id'])
print(f'Sum of person weights: {controls["pers_taz_weight"].sum()}')
print(f"Sum of race cols before processing: {controls[race_cat_list].sum().sum()}")
print(f"Proportion of race cols before processing:\n{(controls[race_cat_list].sum() / controls[race_cat_list].sum().sum()).map('{:.0%}'.format)}")


Sum of person weights: 5788600
Sum of race cols before processing: 5724087
Proportion of race cols before processing:
white_hispanic                    14%
non_white_hispanic                14%
white_non_hispanic                65%
black_non_hispanic                 6%
other_non_hispanic                 0%
asian_non_hispanic                 1%
two_or_more_races_non_hispanic     0%
dtype: object


In [10]:
# TAZs with no simulated persons fall back to regional race shares
regional_prop = p_sum.sum() / p_sum.sum().sum()

# Multiply each race category proportion by the total weight for the TAZ
for category in race_cat_list:
    prop = controls['taz_id'].map(p_prop[category]).fillna(regional_prop[category])
    controls[category] = prop * controls['pers_taz_weight'].fillna(0)

In [11]:
for index, row in controls.iterrows():
    round_list = row[race_cat_list].values.tolist()
    round_list = saferound(round_list,places=0)
    controls.loc[index,race_cat_list] = round_list
    
controls[race_cat_list] = controls[race_cat_list].astype(int)

In [13]:
print(f"Sum of race cols after processing: {controls[race_cat_list].sum().sum()}")
print(f"Proportion of race cols after processing:\n{(controls[race_cat_list].sum() / controls[race_cat_list].sum().sum()).map('{:.0%}'.format)}")


Sum of race cols after processing: 5788600
Proportion of race cols after processing:
white_hispanic                     3%
non_white_hispanic                16%
white_non_hispanic                40%
black_non_hispanic                 6%
other_non_hispanic                 4%
asian_non_hispanic                21%
two_or_more_races_non_hispanic     9%
dtype: object


In [14]:
controls.to_csv('output/data/future_controls_race.csv', index=False)

In [8]:
import tables

# Not a pandas HDFStore: each column is stored as a separate array under /Household and /Person
with tables.open_file('output/output/hh_and_persons.h5', mode='r') as f:
    hh = pd.DataFrame({node.name: node.read() for node in f.root.Household})
    per = pd.DataFrame({node.name: node.read() for node in f.root.Person})


In [9]:
# Same bins/labels as generate_controls.recode; pd.cut default right=True, so prace <= 0 becomes NaN
urbansim_race_bins = [0, 1, 2, 3, 4, 5, 6, 200]
urbansim_race_labels = [
    'white_non_hispanic',
    'black_non_hispanic',
    'asian_non_hispanic',
    'other_non_hispanic',
    'two_or_more_races_non_hispanic',
    'white_hispanic',
    'non_white_hispanic'
]


In [10]:
per['race'] = pd.cut(per['prace'], bins=urbansim_race_bins, labels=urbansim_race_labels)


In [11]:
# value_counts drops NaN, matching crosstab in generate_controls
print(f"Urbansim persons by race {per['race'].value_counts(normalize=True).map('{:.0%}'.format)}")

Urbansim persons by race race
white_non_hispanic                40%
asian_non_hispanic                21%
non_white_hispanic                16%
two_or_more_races_non_hispanic     9%
black_non_hispanic                 6%
other_non_hispanic                 4%
white_hispanic                     3%
Name: proportion, dtype: object


In [19]:
per.groupby(['prace']).agg({'race': 'first'})

,race
prace,
1,white_non_hispanic
2,black_non_hispanic
3,asian_non_hispanic
4,other_non_hispanic
5,two_or_more_races_non_hispanic
6,white_hispanic
7,non_white_hispanic


In [20]:
len(per)

5788048

In [21]:
len(hh)

2421055